In [1]:
import os
import tarfile
import random
import re
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.utils.spectral_norm as spectral_norm
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torchaudio.utils import _download_asset
from torch.utils.data import Dataset, DataLoader, Subset
from jiwer import wer
from tqdm.auto import tqdm
import whisper

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

In [3]:
class Config:
    sr = 16000
    batch_size = 4
    epochs = 20
    lr_g = 1e-4
    lr_d = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tar_path = "../data/raw/ru_train_0.tar"
    tsv_path = "../data/raw/train(1).tsv"
    extract_dir = "../data/raw/ru_train_data"
    n_fft = 512
    hop_length = 256
    win_length = 512
    num_freqs = n_fft // 2 + 1
    T_frames = 192
    
    duration_sec = 3.0
    target_samples = int(sr * duration_sec)

In [4]:
class LearnableSigmoid(nn.Module):
    def __init__(self, in_features: int = 257, beta: float = 1.2):
        super(LearnableSigmoid, self).__init__()
        self.beta = beta
        self.alpha = nn.Parameter(torch.ones(in_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.beta * torch.sigmoid(self.alpha * x)

class Generator(nn.Module):
    def __init__(self, input_dim: int = 257, hidden_dim: int = 200, num_layers: int = 2):
        super(Generator, self).__init__()
        self.blstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc1 = nn.Linear(hidden_dim * 2, 300)
        self.leaky_relu = nn.LeakyReLU()
        self.fc2 = nn.Linear(300, 257)
        self.learnable_sigmoid = LearnableSigmoid(in_features=257, beta=1.2)
        self.mask_lower_bound = 0.05

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out, _ = self.blstm(x)
        out = self.fc1(out)
        out = self.leaky_relu(out)
        out = self.fc2(out)
        mask = self.learnable_sigmoid(out)
        mask = torch.clamp(mask, min=self.mask_lower_bound)
        return x * mask

class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.leaky_relu = nn.LeakyReLU()
        self.conv1 = spectral_norm(nn.Conv2d(2, 15, (5, 5)))
        self.conv2 = spectral_norm(nn.Conv2d(15, 15, (5, 5)))
        self.conv3 = spectral_norm(nn.Conv2d(15, 15, (5, 5)))
        self.conv4 = spectral_norm(nn.Conv2d(15, 15, (5, 5)))
        self.fc1 = spectral_norm(nn.Linear(15, 50))
        self.fc2 = spectral_norm(nn.Linear(50, 10))
        self.fc3 = spectral_norm(nn.Linear(10, 1))

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        if x.dim() == 3: x = x.unsqueeze(1)
        if y.dim() == 3: y = y.unsqueeze(1)
        inp = torch.cat([x, y], dim=1)
        out = self.leaky_relu(self.conv1(inp))
        out = self.leaky_relu(self.conv2(out))
        out = self.leaky_relu(self.conv3(out))
        out = self.leaky_relu(self.conv4(out))
        out = torch.mean(out, dim=(2, 3))
        out = self.leaky_relu(self.fc1(out))
        out = self.leaky_relu(self.fc2(out))
        return self.fc3(out)

In [5]:
def calculate_si_sdr(est, ref):
    eps = 1e-8
    est = est - torch.mean(est, dim=-1, keepdim=True)
    ref = ref - torch.mean(ref, dim=-1, keepdim=True)
    dot = torch.sum(est * ref, dim=-1, keepdim=True)
    ref_pow = torch.sum(ref**2, dim=-1, keepdim=True) + eps
    proj = dot * ref / ref_pow
    noise = est - proj
    si_sdr = 10 * torch.log10(torch.sum(proj**2, dim=-1) / (torch.sum(noise**2, dim=-1) + eps) + eps)
    return torch.clamp((si_sdr + 20) / 50, 0, 1)

def clean_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower()).strip()

def get_snr_scale(signal, noise, snr_db):
    sig_p = signal.norm(p=2)**2 / (signal.numel() + 1e-8)
    noi_p = noise.norm(p=2)**2 / (noise.numel() + 1e-8)
    target_p = sig_p / (10 ** (snr_db / 10))
    return torch.sqrt(target_p / (noi_p + 1e-8))

def apply_noise(clean, force_type=None, file_seed=None):
    if file_seed is not None:
        random.seed(file_seed)
        np.random.seed(file_seed)
        torch.manual_seed(file_seed)

    snr = random.uniform(-5, 15)
    n_len = clean.shape[-1]

    allowed_noises = ['babble', 'rir', 'white']

    if force_type is not None and force_type in allowed_noises:
        noise_cat = force_type
    else:
        noise_cat = random.choice(allowed_noises)

    if noise_cat == 'babble':
        noise = BABBLE_WAVEFORM
        if noise.shape[-1] < n_len:
            repeats = (n_len // noise.shape[-1]) + 2
            noise = noise.repeat(1, repeats)
        max_start = noise.shape[-1] - n_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        noise_crop = noise[:, start:start+n_len]
        scale = get_snr_scale(clean, noise_crop, snr)
        noisy = clean + noise_crop * scale

    elif noise_cat == 'rir':
        rir = RIR_WAVEFORM
        n_fft_conv = n_len + rir.shape[-1] - 1
        clean_fft = torch.fft.rfft(clean, n=n_fft_conv)
        rir_fft = torch.fft.rfft(rir, n=n_fft_conv)
        augmented = torch.fft.irfft(clean_fft * rir_fft, n=n_fft_conv)
        noisy = augmented[:, :n_len]
        white = torch.randn_like(clean)
        scale = get_snr_scale(noisy, white, snr + 10)
        noisy = noisy + white * scale

    elif noise_cat == 'white':
        noise = torch.randn(1, n_len, device=clean.device)
        noise = noise / (noise.abs().max() + 1e-8)
        scale = get_snr_scale(clean, noise, snr)
        noisy = clean + noise * scale

    max_val = noisy.abs().max()
    if max_val > 1.0:
        noisy = noisy / (max_val + 1e-8)

    return noisy

In [6]:
if not os.path.exists(Config.extract_dir):
    os.makedirs(Config.extract_dir, exist_ok=True)
    with tarfile.open(Config.tar_path, "r") as tar: 
        tar.extractall(path=Config.extract_dir)

df_train = pd.read_csv(Config.tsv_path, sep='\t')
reference_dict = {row['path']: clean_text(row['sentence']) for _, row in df_train.iterrows()}

import soundfile as sf
import torch

babble_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-babb-mc01-stu-clo-8000hz.wav")
global BABBLE_WAVEFORM

waveform_np, sr_b = sf.read(babble_path, always_2d=True)
BABBLE_WAVEFORM = torch.tensor(waveform_np.T, dtype=torch.float32)
BABBLE_WAVEFORM = T.Resample(sr_b, Config.sr)(BABBLE_WAVEFORM.mean(dim=0, keepdim=True))

rir_path = _download_asset("tutorial-assets/Lab41-SRI-VOiCES-rm1-impulse-mc01-stu-clo-8000hz.wav")
global RIR_WAVEFORM

waveform_r_np, sr_r = sf.read(rir_path, always_2d=True)
RIR_WAVEFORM = torch.tensor(waveform_r_np.T, dtype=torch.float32)

RIR_WAVEFORM = T.Resample(sr_r, Config.sr)(RIR_WAVEFORM.mean(dim=0, keepdim=True))
RIR_WAVEFORM = RIR_WAVEFORM[:, :int(Config.sr * 0.3)]
RIR_WAVEFORM = RIR_WAVEFORM / torch.norm(RIR_WAVEFORM, p=2)

In [7]:
class WaveformDataset(Dataset):
    def __init__(self, data_dir, ref_dict, is_train=True):
        self.data_dir = data_dir
        self.ref_dict = ref_dict
        self.is_train = is_train
        file_list = []
        for r, _, fs in os.walk(data_dir):
            for f in fs:
                if f.endswith('.mp3') and f in ref_dict:
                    file_list.append(os.path.join(r, f))
        self.files = sorted(file_list)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        wav_np, sr = librosa.load(file_path, sr=None, mono=False)
        wav = torch.from_numpy(wav_np)
        if wav.ndim == 1:
            wav = wav.unsqueeze(0)

        if sr != Config.sr:
            wav = T.Resample(sr, Config.sr)(wav)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        if wav.shape[-1] > Config.target_samples:
            start = random.randint(0, wav.shape[-1] - Config.target_samples) if self.is_train else 0
            wav = wav[:, start:start + Config.target_samples]
        else:
            wav = F.pad(wav, (0, Config.target_samples - wav.shape[-1]))

        clean = wav
        noisy = apply_noise(clean) if self.is_train else clean
        return noisy.squeeze(0), clean.squeeze(0), self.ref_dict[os.path.basename(file_path)]

In [8]:
dataset = WaveformDataset(Config.extract_dir, reference_dict, is_train=True)
train_size = int(0.9 * len(dataset))
generator = torch.Generator().manual_seed(42)
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size],generator = generator)
train_loader = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=True)

generator = Generator().to(Config.device)
discriminator = Discriminator().to(Config.device)
opt_g = torch.optim.Adam(generator.parameters(), lr=Config.lr_g)
opt_d = torch.optim.Adam(discriminator.parameters(), lr=Config.lr_d)
criterion = nn.MSELoss()
window = torch.hann_window(Config.n_fft).to(Config.device)

In [ ]:
for epoch in range(1, Config.epochs + 1):
    generator.train()
    discriminator.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}")
    
    for noisy, clean, _ in pbar:
        noisy, clean = noisy.to(Config.device), clean.to(Config.device)
        X = torch.stft(noisy, Config.n_fft, Config.hop_length, window=window, return_complex=True, center=True)
        S = torch.stft(clean, Config.n_fft, Config.hop_length, window=window, return_complex=True, center=True)
        X_mag, S_mag = torch.abs(X).transpose(1, 2), torch.abs(S).transpose(1, 2)
        
        enhanced_mag = generator(X_mag)
        enhanced_complex = enhanced_mag.transpose(1, 2) * torch.exp(1j * torch.angle(X))
        enhanced_wav = torch.istft(enhanced_complex, Config.n_fft, Config.hop_length, window=window, length=noisy.shape[1])

        opt_d.zero_grad()
        score_c = discriminator(S_mag, S_mag)
        score_e = discriminator(enhanced_mag.detach(), S_mag)
        score_n = discriminator(X_mag, S_mag)
        
        loss_d = criterion(score_c, torch.ones_like(score_c)) + \
                 criterion(score_e, calculate_si_sdr(enhanced_wav.detach(), clean).unsqueeze(1)) + \
                 criterion(score_n, calculate_si_sdr(noisy, clean).unsqueeze(1))
        
        loss_d.backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
        opt_d.step()

        opt_g.zero_grad()
        loss_g = criterion(discriminator(enhanced_mag, S_mag), torch.ones_like(score_c))
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
        opt_g.step()

        if epoch % 5 == 0:
            torch.save(generator.state_dict(), f"metricgan_si_sdr_{epoch}.pth")
        
        pbar.set_postfix({"L_G": f"{loss_g.item():.4f}", "L_D": f"{loss_d.item():.4f}"})

torch.save(generator.state_dict(), "../models/metricgan_weights.pth")

Epoch 1:   0%|          | 8/5954 [00:05<1:12:56,  1.36it/s, L_G=0.7303, L_D=0.8946]


KeyboardInterrupt: 

In [11]:
generator.load_state_dict(torch.load('../models/metricgan_weights.pth', map_location='cpu'))

<All keys matched successfully>

In [12]:
seed_everything(42)

In [13]:
def evaluate(model, device, ds, limit=20):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    noise_types = ['babble', 'rir', 'white']
    stats = {n: {"wer_n": [], "wer_d": []} for n in noise_types}
    win = torch.hann_window(Config.n_fft).to(device)

    if limit is not None:
        indices = list(range(min(limit, len(ds))))
    else:
        indices = list(range(len(ds)))

    with torch.no_grad():
        for idx in tqdm(indices, desc="WER Eval"):
            _, c_wav, ref = ds[idx]
            ref = clean_text(ref)
            
            for nt in noise_types:
                n_wav = apply_noise(c_wav.unsqueeze(0), force_type=nt, file_seed=idx).to(device)
                
                X = torch.stft(n_wav.squeeze(1), Config.n_fft, Config.hop_length, window=win, return_complex=True)
                E_mag = model(torch.abs(X).transpose(1, 2))
                E_wav = torch.istft(E_mag.transpose(1, 2) * torch.exp(1j * torch.angle(X)), Config.n_fft, Config.hop_length, window=win, length=n_wav.shape[-1])

                E_wav = torch.nan_to_num(E_wav, nan=0.0, posinf=1.0, neginf=-1.0)
                n_wav = torch.nan_to_num(n_wav, nan=0.0, posinf=1.0, neginf=-1.0)

                t_n = asr.transcribe(n_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']
                t_d = asr.transcribe(E_wav.squeeze().cpu().numpy(), fp16=False, language='ru')['text']

                stats[nt]["wer_n"].append(wer(ref, clean_text(t_n)))
                stats[nt]["wer_d"].append(wer(ref, clean_text(t_d)))

    print(f"\n{'Noise Type':<10} | {'WER Noisy':<10} | {'WER Denoised':<10}")
    for n in noise_types:
        wn, wd = np.mean(stats[n]['wer_n']), np.mean(stats[n]['wer_d'])
        print(f"{n:<10} | {wn:<10.4f} | {wd:<10.4f}")

evaluate(generator, Config.device, val_ds, limit=20)

c:\Users\itism\Desktop\test\ml-asr\.venv\Lib\site-packages\whisper\__init__.py:69: UserWarning: C:\Users\itism\.cache\whisper\large-v3.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(
  3%|█                                    | 87.9M/2.88G [00:08<04:24, 11.3MiB/s]


KeyboardInterrupt: 

In [ ]:
from jiwer import process_words

def evaluate_and_listen_components(model, device, val_dataset, limit=None):
    asr = whisper.load_model("large-v3").to(device)
    model.eval()
    
    noise_types = ['babble', 'rir', 'white']
    
    stats = {n: {
        "wer_n": [], "s_n": [], "d_n": [], "i_n": [],
        "wer_d": [], "s_d": [], "d_d": [], "i_d": []
    } for n in noise_types}
    
    win = torch.hann_window(Config.n_fft).to(device)
    
    if limit is not None:
        indices = list(range(min(limit, len(val_dataset))))
    else:
        indices = list(range(len(val_dataset)))
    
    with torch.no_grad():
        for idx in tqdm(indices, desc="Evaluation (Macro-average)"):
            _, clean_wav, ref_text = val_dataset[idx]
            clean_tensor_cpu = clean_wav.unsqueeze(0)
    
            for n_type in noise_types:
                n_wav = apply_noise(clean_tensor_cpu, force_type=n_type, file_seed=idx).to(device)
                
                X = torch.stft(n_wav.squeeze(1), Config.n_fft, Config.hop_length, window=win, return_complex=True)
                E_mag = model(torch.abs(X).transpose(1, 2))
                E_wav = torch.istft(E_mag.transpose(1, 2) * torch.exp(1j * torch.angle(X)), Config.n_fft, Config.hop_length, window=win, length=n_wav.shape[-1])
                
                E_wav = torch.nan_to_num(E_wav, nan=0.0, posinf=1.0, neginf=-1.0)
                n_wav = torch.nan_to_num(n_wav, nan=0.0, posinf=1.0, neginf=-1.0)
    
                noisy_np = n_wav.squeeze().cpu().numpy()
                denoised_np = E_wav.squeeze().cpu().numpy()
    
                t_n = asr.transcribe(noisy_np, fp16=False, language='ru')['text']
                t_d = asr.transcribe(denoised_np, fp16=False, language='ru')['text']
    
                clean_ref = clean_text(ref_text)
                clean_hyp_n = clean_text(t_n)
                clean_hyp_d = clean_text(t_d)
    
                out_n = process_words(clean_ref, clean_hyp_n)
                n_words_n = out_n.substitutions + out_n.deletions + out_n.hits
                
                if n_words_n > 0:
                    stats[n_type]["wer_n"].append((out_n.substitutions + out_n.deletions + out_n.insertions) / n_words_n)
                    stats[n_type]["s_n"].append(out_n.substitutions / n_words_n)
                    stats[n_type]["d_n"].append(out_n.deletions / n_words_n)
                    stats[n_type]["i_n"].append(out_n.insertions / n_words_n)
                else:
                    stats[n_type]["wer_n"].append(0.0)
                    stats[n_type]["s_n"].append(0.0)
                    stats[n_type]["d_n"].append(0.0)
                    stats[n_type]["i_n"].append(0.0)
    
                out_d = process_words(clean_ref, clean_hyp_d)
                n_words_d = out_d.substitutions + out_d.deletions + out_d.hits
                
                if n_words_d > 0:
                    stats[n_type]["wer_d"].append((out_d.substitutions + out_d.deletions + out_d.insertions) / n_words_d)
                    stats[n_type]["s_d"].append(out_d.substitutions / n_words_d)
                    stats[n_type]["d_d"].append(out_d.deletions / n_words_d)
                    stats[n_type]["i_d"].append(out_d.insertions / n_words_d)
                else:
                    stats[n_type]["wer_d"].append(0.0)
                    stats[n_type]["s_d"].append(0.0)
                    stats[n_type]["d_d"].append(0.0)
                    stats[n_type]["i_d"].append(0.0)
    
    header = f"{'Noise':<8} | {'WER_N':<7} (S/D/I) | {'WER_D':<7} (S/D/I) | {'Gain WER':<8}"
    print(header)
    print("-" * 75)
    
    for n_type in noise_types:
        wer_n = np.mean(stats[n_type]["wer_n"])
        s_n_pct = np.mean(stats[n_type]["s_n"])
        d_n_pct = np.mean(stats[n_type]["d_n"])
        i_n_pct = np.mean(stats[n_type]["i_n"])
    
        wer_d = np.mean(stats[n_type]["wer_d"])
        s_d_pct = np.mean(stats[n_type]["s_d"])
        d_d_pct = np.mean(stats[n_type]["d_d"])
        i_d_pct = np.mean(stats[n_type]["i_d"])
    
        str_noisy = f"{wer_n:.4f} ({s_n_pct:.4f}/{d_n_pct:.4f}/{i_n_pct:.4f})"
        str_denois = f"{wer_d:.4f} ({s_d_pct:.4f}/{d_d_pct:.4f}/{i_d_pct:.4f})"
    
        print(f"{n_type:<8} | {str_noisy:<27} | {str_denois:<27} | {wer_n - wer_d:<8.4f}")

evaluate_and_listen_components(generator, Config.device, val_ds, limit=20)

Evaluation (Macro-average):   0%|          | 0/20 [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchaudio/_backend/ffmpeg.py:88: UserWarning: torio.io._streaming_media_decoder.StreamingMediaDecoder has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be r

Noise    | WER_N   (S/D/I) | WER_D   (S/D/I) | Gain WER
---------------------------------------------------------------------------
babble   | 0.5376 (0.1392/0.3875/0.0108) | 0.4547 (0.0957/0.3545/0.0045) | 0.0828  
rir      | 1.0000 (0.3116/0.6884/0.0000) | 1.0250 (0.3338/0.6662/0.0250) | -0.0250 
white    | 0.5670 (0.2060/0.3548/0.0063) | 0.6437 (0.2306/0.4086/0.0045) | -0.0767 
